In [9]:
import pyvista as pv
import numpy as np

In [10]:
import pyvista as pv
import numpy as np


def get_damage_slice_bottom(
    xdmf_file,
    damage_name="damage",
    damage_threshold=0.5,
    timestep=-1,
):
    """
    Find the bottom-most point of a damage iso-surface
    from a time-dependent XDMF file.

    Parameters
    ----------
    xdmf_file : str
        Path to the XDMF file.

    damage_name : str
        Name of the CG1 damage field stored as point data.

    damage_threshold : float
        Damage value defining the iso-surface.

    timestep : int
        Timestep index to use.
        -1 = last timestep
         0 = first timestep
         1 = second timestep
         etc.

    Returns
    -------
    zmin : float
        Minimum z-coordinate of the damage iso-surface.

    bottom_point : ndarray
        [x, y, z] coordinates of the bottom-most point.

    time_value : float
        Actual simulation time corresponding to the selected timestep.
    """

    # --------------------------------------------------
    # Open time-dependent XDMF
    # --------------------------------------------------
    reader = pv.get_reader(xdmf_file)

    # Available simulation times
    times = np.asarray(reader.time_values)

    if len(times) == 0:
        raise ValueError("No timesteps found in XDMF file.")

    # Allow Python-style negative indexing
    try:
        time_value = times[timestep]
    except IndexError:
        raise IndexError(
            "Requested timestep {} but file contains {} timesteps.".format(
                timestep, len(times)
            )
        )

    # --------------------------------------------------
    # Select timestep
    # --------------------------------------------------
    reader.set_active_time_value(float(time_value))

    # Read selected timestep
    grid = reader.read()

    # --------------------------------------------------
    # Sometimes XDMF produces a MultiBlock
    # --------------------------------------------------
    if isinstance(grid, pv.MultiBlock):
        grid = grid.combine()

    # --------------------------------------------------
    # Check damage field
    # --------------------------------------------------
    if damage_name not in grid.point_data:
        raise ValueError(
            "Damage field '{}' not found.\n"
            "Available point fields: {}".format(
                damage_name,
                list(grid.point_data.keys()),
            )
        )

    grid.set_active_scalars(
        damage_name,
        preference="point",
    )

    # --------------------------------------------------
    # Construct damage iso-surface through elements
    # --------------------------------------------------
    surface = grid.contour(
        isosurfaces=[damage_threshold],
        scalars=damage_name,
    )

    if surface.n_points == 0:
        raise ValueError(
            "No damage={} iso-surface found at timestep {} "
            "(time={}).".format(
                damage_threshold,
                timestep,
                time_value,
            )
        )

    # --------------------------------------------------
    # Bottom-most point
    # --------------------------------------------------
    i = np.argmin(surface.points[:, 2])

    bottom_point = surface.points[i].copy()
    zmin = float(bottom_point[2])

    return zmin, bottom_point, float(time_value)

In [11]:
import glob
import os

xdmf_files = glob.glob(os.path.join("**", "*.xdmf"), recursive=True)
xdmf_files.sort(key=lambda x: os.path.basename(os.path.dirname(x)))
for xdmf_file in xdmf_files:
    zmin, point, time = get_damage_slice_bottom(
        xdmf_file,
        damage_name="damage",
        damage_threshold=0.6,
        timestep=-1,
    )
    print(f"Study-{os.path.basename(os.path.dirname(xdmf_file))} with crack cord {zmin:.2f}m with crack depth {125-zmin:.2f}m")

Study-01 with crack cord 83.32m with crack depth 41.68m
Study-02 with crack cord 35.30m with crack depth 89.70m
Study-03 with crack cord 16.68m with crack depth 108.32m
Study-04 with crack cord 85.09m with crack depth 39.91m
Study-05 with crack cord 36.98m with crack depth 88.02m
Study-06 with crack cord 13.72m with crack depth 111.28m


In [12]:
86,36,13

(86, 36, 13)